# State-teacher cost on a T4

**What this answers.** Whether the frozen speaker-state teacher
(decisions-pending.md D14) is affordable as a training term, and if not, how
short a segment it has to score to become affordable.

**Why it cannot be answered on the laptop.** Every timing recorded for the
teacher so far is CPU. Training is a Kaggle T4. Scoring the full 4.008 s chunk
is 13 one-second windows per example, forward AND backward through a 21 M
parameter frozen encoder, against a current 0.674 s/step at batch 3. A 16-epoch
run takes 10.5 h against a 12 h session cap, so the budget is about **+14 % per
step**.

**Setup.** Accelerator **GPU T4 x2**. Internet off is fine. Add the dataset
`tse-code-state-teacher` (built by `scripts/make_kaggle_bundle.py --split sir0 --code-only`).

**No audio dataset is needed.** The profiler uses synthetic noise -- audio
content does not affect timing -- so the ~30 GB corpus is skipped entirely.

In [ ]:
!pip install -q speechbrain==1.1.1

In [ ]:
import shutil, pathlib, subprocess

# OVERWRITE, never skip. /kaggle/working PERSISTS between runs in a session, so
# an "if it exists, skip" copy silently keeps the PREVIOUS run's code after the
# dataset is updated -- which happened on 2026-09-11: two runs produced the same
# traceback from a file that had already been fixed, and the line numbers in the
# traceback were the only evidence.
SRC = pathlib.Path("/kaggle/input/tse-code-state-teacher")
DST = pathlib.Path("/kaggle/working")
for item in SRC.iterdir():
    target = DST / item.name
    if target.is_dir():
        shutil.rmtree(target)
    elif target.exists():
        target.unlink()
    (shutil.copytree if item.is_dir() else shutil.copy2)(item, target)

%cd /kaggle/working
print("bundle commit:",
      pathlib.Path("docs/bundle_commit.txt").read_text().strip())

# Prove the copy took. If these two lines are absent, the dataset attached to
# this notebook is an older VERSION -- check the Data panel on the right, not
# just the dataset name.
source = pathlib.Path("scripts/profile_state_teacher.py").read_text()
assert "scaler.scale(loss).backward()" in source, \
    "profile_state_teacher.py is the pre-AMP version -- attach the newer dataset version"
assert "self.head.train()" in pathlib.Path("src/models/state_teacher.py").read_text(), \
    "state_teacher.py lacks the cuDNN train-mode fix -- attach the newer dataset version"
print("code version checks passed")

### The one check worth making before anything runs

`embedding_model.ckpt` must be **~83 MB**. If it is ~150 bytes it is still a
symlink into a HuggingFace cache that does not exist here, and the teacher will
fail to load with a confusing error about a missing path.

`make_kaggle_bundle.py` dereferences and re-hashes it against the hashes stored
inside the teacher checkpoint, so this should never fail -- but it costs a
second and the alternative is losing the session to it.

In [ ]:
import pathlib
for f in sorted(pathlib.Path("ecapa_pretrained").iterdir()):
    print(f"  {f.name:26s} {f.stat().st_size / 2**20:8.2f} MB")
assert (pathlib.Path("ecapa_pretrained/embedding_model.ckpt").stat().st_size
        > 10 * 2**20), "backbone is still a symlink -- rebuild the bundle"

import torch
print(f"\ncuda {torch.cuda.is_available()}   "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''}")
print(f"devices visible: {torch.cuda.device_count()}")

### The measurement

Sweeps the length of the contiguous segment handed to the teacher, from 1.0 s
(1 window) to the full 4.008 s (13 windows), and reports per length:

| | |
|---|---|
| **seconds per step** | forward, loss, backward, optimiser |
| **peak GPU memory** | against the ~15 GB card |
| **`L_state`** | what the teacher actually says |

**Why a contiguous segment and not a random subset of windows.** The term is a
mean over windows, so a random subset would be an unbiased estimate of it --
the mini-batch argument. But the teacher's head is a BiLSTM over the window
sequence, so a loss on 4 windows still needs all 13 embeddings to feed the
recurrence, and the backward flows through it to all 13 anyway. Shortening the
SEQUENCE shortens both.

**Why `L_state` is reported alongside the timing.** A shorter segment gives the
head less context than the 13 windows it was fitted on. A setting that is cheap
AND changes what the teacher says is not a saving -- it is a different teacher.

In [ ]:
!python scripts/profile_state_teacher.py \
    --teacher models/state_detector_notebook.pt \
    --ecapa-dir ecapa_pretrained \
    --device cuda

### Reading the figure

**Left panel: what is affordable.** Take the largest window count whose point
sits under the 12 h session cap.

**Right panel: whether that is honest.** `L_state` should be roughly flat. Drift
means the shorter context changed the measurement.

**The answer is the largest window count that fits on the left and does not move
on the right.** If all 13 fit, take 13 and the subsampling question disappears.

Then set `data.chunk_s`-worth of segment in the arm's config and record the
choice in `decisions-pending.md` D14 with this figure beside it.

In [ ]:
from IPython.display import Image, display
import glob, json

out = sorted(glob.glob("experiments/results/*-state-teacher-cost"))[-1]
display(Image(filename=f"{out}/state_teacher_cost.png"))

results = json.load(open(f"{out}/results.json"))
baseline = results["baseline_seconds_per_step"]
print(f"{results['gpu']}   batch {results['batch_examples']} examples   "
      f"baseline {baseline:.3f} s/step\n")
print(f"{'windows':>8} {'s/step':>9} {'overhead':>10} {'16-ep hours':>12} "
      f"{'peak GB':>9} {'L_state':>9}")
for row in results["rows"]:
    if row["n_windows"] == 0:
        continue
    hours = 10.5 * row["seconds_per_step"] / baseline
    flag = "" if hours <= 12 else "   OVER CAP"
    print(f"{row['n_windows']:>8} {row['seconds_per_step']:>9.3f} "
          f"{row['overhead_pct']:>9.0f}% {hours:>12.1f} {row['peak_gb']:>9.2f} "
          f"{row['L_state']:>9.4f}{flag}")

### Download before the session ends

Both files are the deliverable: the figure goes in the report, the JSON carries
the GPU name, the commit stamp and every number behind it.

In [ ]:
import glob, shutil
out = sorted(glob.glob("experiments/results/*-state-teacher-cost"))[-1]
for name in ("state_teacher_cost.pdf", "state_teacher_cost.png", "results.json"):
    shutil.copy2(f"{out}/{name}", f"/kaggle/working/{name}")
    print(f"  /kaggle/working/{name}")